# Real-Time Driver Drowsiness Detection: CNN + LSTM Model Training
**Course**: Final Year Engineering Capstone Project
**Objective**: Train a multi-class temporal deep learning network to classify driver state (*Alert*, *Mild Drowsiness*, *Severe Drowsiness*) using facial landmark sequences and CNN-extracted facial features.

In [ ]:
# 1. Environment & Package Imports
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

## 2. Model Architecture: TimeDistributed CNN + Bidirectional LSTM
The spatial CNN extracts periocular features from individual face frames (112x112x3), while the Bi-LSTM layers model sequential patterns over a 20-frame window.

In [ ]:
def build_model(seq_len=20, img_shape=(112, 112, 3), num_classes=3):
    # CNN Backbone
    cnn_input = layers.Input(shape=img_shape)
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(cnn_input)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu', name='target_conv_layer')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    cnn_backbone = models.Model(cnn_input, x, name='spatial_cnn')

    # Temporal Model
    seq_input = layers.Input(shape=(seq_len, *img_shape))
    spatial_seq = layers.TimeDistributed(cnn_backbone)(seq_input)
    lstm = layers.Bidirectional(layers.LSTM(64, return_sequences=False, dropout=0.3))(spatial_seq)
    dense = layers.Dense(64, activation='relu')(lstm)
    dense = layers.Dropout(0.4)(dense)
    output = layers.Dense(num_classes, activation='softmax')(dense)

    model = models.Model(seq_input, output, name='CNN_LSTM_Drowsiness')
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = build_model()
model.summary()

In [ ]:
# 3. Training & Evaluation
# callbacks = [tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
# history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=25, callbacks=callbacks)
print("Model successfully configured for training on webcam dataset.")